# Tutorial 09: Multiple attention heads and position

**Prerequisite:** lecture 09. **Compute:** CPU; standard library and NumPy.

Predict the result before each code cell. Run cells in order from a fresh kernel. The course bootstrap needs the full extracted course pack. Outputs are intentionally empty; keep your actual measurements.


In [ ]:
from pathlib import Path
import sys,json,time,random,hashlib,math
if hasattr(sys.stdout,'reconfigure'): sys.stdout.reconfigure(encoding='utf-8')
candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'llm_course',Path.cwd()/'llm_course_v2',Path('/content/llm_course'),Path('/content/llm_course_v2')]
COURSE=next((p for p in candidates if (p/'coursekit/core.py').exists()),None)
if COURSE is None: raise RuntimeError('Extract the complete course pack; run from its root or notebooks directory.')
if str(COURSE) not in sys.path:sys.path.insert(0,str(COURSE))
RUNS=COURSE/'runs';RUNS.mkdir(exist_ok=True)
from coursekit.core import *
random.seed(7)
print('Course root:',COURSE)


A multi-head layer divides projected features into parallel heads. Reshaping does not learn anything. The projection matrices do. Here we inspect the shape transformation, then show how positional vectors distinguish otherwise identical tokens.


In [ ]:
import numpy as np
B,T,d_model,H=1,6,8,2;d_head=d_model//H
x=np.arange(B*T*d_model).reshape(B,T,d_model)
heads=x.reshape(B,T,H,d_head).transpose(0,2,1,3)
merged=heads.transpose(0,2,1,3).reshape(B,T,d_model)
print('Original',x.shape,'split',heads.shape,'merged',merged.shape)
assert np.array_equal(x,merged)
pos=np.arange(T)[:,None];dims=np.arange(0,d_model,2)
angles=pos/(10000**(dims/d_model))
pe=np.zeros((T,d_model));pe[:,0::2]=np.sin(angles);pe[:,1::2]=np.cos(angles)
print('Position vectors:',pe[:3])
assert not np.allclose(pe[0],pe[1])


### Pause before the next step
Track the location of feature six at position two through the split and merge.


### RoPE with different position angles
Compare angle pairs (30°,90°) and (60°,120°). Both have the same relative offset. This two-coordinate calculation illustrates relative-position dependence; it does not by itself establish long-context reliability.

In [ ]:
import math
def rotate(v,theta):
    c,s=math.cos(theta),math.sin(theta);return [v[0]*c-v[1]*s,v[0]*s+v[1]*c]
def dp(a,b):return sum(x*y for x,y in zip(a,b))
q=k=[1.,0.]
for q_deg,k_deg in [(0,0),(30,90),(60,120)]:
    value=dp(rotate(q,math.radians(q_deg)),rotate(k,math.radians(k_deg)))
    expected=math.cos(math.radians(k_deg-q_deg))
    assert math.isclose(value,expected,abs_tol=1e-12)
    print(q_deg,k_deg,'relative offset',k_deg-q_deg,'dot',value)


### Independent experiment
Implement one attention calculation per head, then concatenate. Compare with a single head of dimension eight, using distinct but recorded matrices. Discuss what is an architectural difference and what is merely a layout transformation.

Submit a short report containing your prediction, the actual result, a comparison and a limitation. Record any environment or data changes. Do not substitute an illustrative number for a measured result.
